In [1]:
import json
import requests
import os

class AccessibilityChecker:
    def __init__(self, json_file, download_dir="downloads"):
        self.metadata_repository = self.load_metadata(json_file)
        self.download_dir = download_dir
        os.makedirs(self.download_dir, exist_ok=True)  # Ensure download directory exists

    def load_metadata(self, json_file):
        with open(json_file, 'r') as file:
            data = json.load(file)
        return {artifact_id: artifact['uri'] for artifact_id, artifact in data['artifacts'].items()}

    def is_accessible(self, uri):
        if not uri:
            return None
        try:
            response = requests.get(uri, allow_redirects=True, timeout=10)
            if response.status_code == 200:
                return response.content  # Return the downloaded artifact data
            else:
                return None
        except requests.RequestException:
            return None

    def save_artifact(self, artifact_id, data):
        file_path = os.path.join(self.download_dir, f"{artifact_id}.bin")
        with open(file_path, "wb") as file:
            file.write(data)
        print(f"Artifact {artifact_id} saved to {file_path}")

    def check_accessibility(self):
        for artifact_id, uri in self.metadata_repository.items():
            artifact = self.is_accessible(uri)
            if artifact:
                print(f"Artifact {artifact_id} is accessible and retrieved.")
                self.save_artifact(artifact_id, artifact)
                if len(artifact) < 500:  # Print preview for small text-based artifacts
                    print(f"Preview of {artifact_id}: {artifact.decode(errors='ignore')[:200]}")
            else:
                print(f"Artifact {artifact_id} is NOT accessible.")

if __name__ == "__main__":
    json_file_path = 'artifacts.json'
    checker = AccessibilityChecker(json_file_path)
    checker.check_accessibility()


Artifact artifact_1 is accessible and retrieved.
Artifact artifact_1 saved to downloads\artifact_1.bin
Artifact artifact_2 is NOT accessible.
